Initialize **Objective**

In [1]:
import numpy as np
from modopt import Problem

class Rosenbrock2D(Problem):
    def initialize(self, ):
        # Name your problem
        self.problem_name = 'Rosenbrock2D'

    def setup(self):
        # Add design variables of your problem
        self.add_design_variables('x',
                                  shape=(2, ),
                                  vals=np.array([.3, .3]))
        self.add_objective('f')

    def setup_derivatives(self):
        # Declare objective gradient and its shape
        self.declare_objective_gradient(wrt='x', )

    # Compute the value of the objective with given design variable values
    def compute_objective(self, dvs, obj):
        x, y = dvs['x']
        obj['f'] = (1-x)**2 + 100*(y-x**2)**2

    def compute_objective_gradient(self, dvs, grad):
        x = dvs['x']
        grad['x'] = np.array([
            -400 * x[0] * (x[1] - x[0]**2) + 2 * (x[0] - 1),
            200 * (x[1] - x[0]**2)
        ])

Initialize **Optimizer**

In [12]:
import numpy as np
import time
from modopt import Optimizer
from scipy.stats.qmc import Sobol

class SimpleGA(Optimizer):


    def initialize(self):
        # Name your algorithm
        self.solver_name = 'Simple_Genetic_Algorithm'

        self.obj = self.problem._compute_objective
        self.grad = self.problem._compute_objective_gradient

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default = 2**9, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default= 4.0, types = float)
        self.options.declare('mutationRate', default = 0.2, types = float)
        self.options.declare('mutationStd', default = self.options['rangeHigh'] - self.options['rangeHigh'], types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        # Instantiate any modules you need for the algorithm
        pass

    def Populate(self):
        n = self.options['initialPopulationSize']

        # Generate Sobol sequence in [0,1]
        qrng = Sobol(d=self.problem.nx, scramble=True)
        matrix = qrng.random(n)

        # Scale to [-4,4]
        population = self.options['rangeLow'] + (self.options['rangeHigh']- self.options['rangeLow']) * matrix
        return population  
    
    def Evaluate(self, population):
        return np.array([self.obj(ind) for ind in population])

    def Fitness_Selection(self,population):
        N, D = population.shape 
        Evaluation = self.Evaluate(population)
        
        population_with_scores = np.column_stack((population, Evaluation))
        population_with_scores = population_with_scores[population_with_scores[:, -1].argsort()]
        
        num_elites = N // 2  # Select half the population
        tournament_size = max(2, N // 5)  # Set tournament size

        selected_indices = []
        for _ in range(num_elites):
            tournament_contestants = np.random.choice(N, tournament_size, replace=False)
            best_index = tournament_contestants[np.argmin(Evaluation[tournament_contestants])]
            selected_indices.append(best_index)

        elites = population[selected_indices]

        return elites 

    def Crossover(self,elites):
        np.random.shuffle(elites)
        N = len(elites)

        offspring = []

        for i in range(0,N,2):
            if i + 1 < N:
                parent1, parent2 = elites[i], elites[i+1]

                var_index = np.random.choice([0,1])

                alpha1 = np.random.uniform(0, 1)
                alpha2 = np.random.uniform(0, 1)

                child1, child2 = parent1.copy(), parent2.copy()
                child1[var_index] = alpha1 * parent1[var_index] + (1 - alpha1) * parent2[var_index]
                child2[var_index] = alpha2 * parent1[var_index] + (1 - alpha2) * parent2[var_index]

                offspring.extend([child1, child2])

        return offspring

    def Mutation(self, elites, offspring):
        newPopulation = np.vstack((elites,offspring))

        N, D = newPopulation.shape

        mutation_mask = np.random.rand(N, D) < self.options['mutationRate']

        gaussian_noise = np.random.normal(0, self.options['mutationStd'], (N, D))
        mutatedPopulation = np.copy(newPopulation)
        mutatedPopulation += mutation_mask * gaussian_noise

        mutatedPopulation = np.clip(mutatedPopulation, self.options['rangeLow'], self.options['rangeHigh'])

        return newPopulation

    def solve(self):
        nx = self.problem.nx
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        population = self.Populate()

        while (opt > opt_tol and itr < maxiter):
            itr_start = time.time()
            itr += 1
            elites = self.Fitness_Selection(population)

            offspring = self.Crossover(elites)

            population = self.Mutation(elites,offspring)

            final_evaluations = self.Evaluate(population)
            optimal_index = np.argmin(final_evaluations)
            f_k = final_evaluations[optimal_index]
            opt = f_k
            x_k = population[optimal_index]

            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k,
                                obj=f_k,
                                opt=opt,
                                time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

In [13]:
import time
from modopt import Optimizer
from deap import base, creator, tools
import random

class DEAPGeneric(Optimizer):

    def initialize(self):

        # Name your algorithm
        self.solver_name = 'DEAP_Genetic_Algorithm'

        self.obj = self.problem._compute_objective
        self.grad = self.problem._compute_objective_gradient

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default =  4.0, types = float)
        self.options.declare('mutationRateGene', default = 0.1, types = float)
        self.options.declare('alpha', default = 0.5, types = float)
        self.options.declare('tournsize', default = 3, types = int)
        self.options.declare('cxProb' , default =  0.5, types = float)
        self.options.declare('mutationRateInd', default =   0.2, types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # -1.0 Weight means minimization, 1.0 for Maximization
            creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attribute", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])  # Adjusted range
        self.toolbox.register("individual", tools.initRepeat, creator.Individual,
                 self.toolbox.attribute, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=self.options['alpha'])
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow'])*0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selTournament, tournsize=self.options['tournsize'])
        def modopt_evaluate(individual):
            return (self.problem._compute_objective(individual),)

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate the entire population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            # Select the next generation individuals
            offspring = self.toolbox.select(pop, len(pop))
            # Clone the selected individuals
            offspring = list(map(self.toolbox.clone, offspring))

            # Apply crossover and mutation on the offspring
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate the individuals with an invalid fitness
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = map(self.toolbox.evaluate, invalid_ind)
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit

            # The population is entirely replaced by the offspring
            pop[:] = offspring

            val = self.obj(pop[0])                # Might be float (SO) or array/tuple (MO)
            val_arr = np.atleast_1d(val)          # Ensures at least 1D
            
            if len(val_arr) == 1:
                best_ind = tools.selBest(pop, 1)[0]  # Single-objective
            else:
                best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
                best_ind = random.choice(best_inds)  # Multi-objective


            f_k = best_ind.fitness.values[0]
            opt = f_k

            x_k = np.array(best_ind)

            itr += 1
            
            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k,
                                obj=f_k,
                                opt=opt,
                                time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

In [4]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

class G3PCX(Optimizer):


    def initialize(self):

        # Name your algorithm
        self.solver_name = 'DEAP_G3PCX'

        self.obj = self.problem._compute_objective
        self.grad = self.problem._compute_objective_gradient

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default =  4.0, types = float)
        self.options.declare('mutationRateGene', default = 0.1, types = float)
        self.options.declare('alpha', default = 0.5, types = float)
        self.options.declare('tournsize', default = 3, types = int)
        self.options.declare('cxProb' , default =  0.9, types = float)
        self.options.declare('mutationRateInd', default =   0.2, types = float)
        self.options.declare('zeta', default=0.1, types=float)
        self.options.declare('eta', default=0.1, types=float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }
    @staticmethod
    def cxPCX(ind1, ind2, zeta, eta):
        """Parent-Centric Crossover (PCX)
        - `zeta`: Scale for offspring deviation
        - `eta`: Scale for perpendicular vector
        """
        p1, p2 = np.array(ind1), np.array(ind2)
        mean_p = 0.5 * (p1 + p2)  # Mean vector
        diff = p2 - p1           # Difference vector

        # Generate offspring along the difference vector
        offspring = mean_p + zeta * diff + eta * np.random.randn(len(p1))
        
        # Convert offspring array to DEAP Individual
        child = creator.Individual(offspring.tolist())
        return (child,) 

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # -1.0 Weight means minimization, 1.0 for Maximization
            creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attr_float", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])
        self.toolbox.register("individual", tools.initRepeat, creator.Individual,
                 self.toolbox.attr_float, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", lambda ind1, ind2: self.cxPCX(ind1, ind2, self.options['zeta'], self.options['eta']))
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow'])*0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selTournament, tournsize=self.options['tournsize'])
        def modopt_evaluate(individual):
            return (self.problem._compute_objective(individual),)

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate the entire population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            # Select the next generation individuals
            offspring = list(map(self.toolbox.clone, pop))

            # Apply crossover and mutation on the offspring
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    offspring_pair = self.toolbox.mate(child1, child2)
                    offspring[offspring.index(child1)] = offspring_pair[0]

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate the individuals with an invalid fitness
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # The population is entirely replaced by the offspring
            pop[:] = self.toolbox.select(pop + offspring, len(pop))

            best_ind = tools.selBest(pop, 1)[0]
            x_k = np.array(best_ind)
            f_k = best_ind.fitness.values[0]
            opt = f_k

            itr += 1
            
            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k.tolist(),
                                obj=f_k,
                                opt=opt,
                                time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

In [6]:
import time
import random
import numpy as np
import cma  # You need to have the 'cma' package installed (pip install cma)
from modopt import Optimizer

class CMAES(Optimizer):
    def initialize(self):
        """Initialize CMA-ES parameters and declare options."""
        self.solver_name = "CMA-ES"
        # Assume a single-objective function that returns a tuple (we use index 0)
        self.obj = self.problem._compute_objective  
        self.options.declare('maxiter', default=300, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('init_sigma', default=0.5, types=float)
        self.options.declare('popsize', default=50, types=int)
        
        # Ensure modOpt's output options are declared
        self.options.declare('readable_outputs', types=list, default=[])
        self.available_outputs = {
            'itr': int,
            'obj': float,
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        """Set up the CMA-ES strategy using the cma package."""
        dim = self.problem.nx
        # If no initial guess is provided, default to a zero vector.
        if self.problem.x0 is None:
            self.problem.x0 = np.zeros(dim)
        # Initialize CMA-ES; note that CMAEvolutionStrategy expects a list.
        self.es = cma.CMAEvolutionStrategy(self.problem.x0.tolist(),
                                           self.options['init_sigma'],
                                           {'popsize': self.options['popsize']})

    def solve(self):
        """Run the CMA-ES optimization."""
        start_time = time.time()
        itr = 0
        best_f = float('inf')
        best_x = None

        # Main optimization loop
        while not self.es.stop() and itr < self.options['maxiter']:
            # Generate a batch of candidate solutions
            solutions = self.es.ask()
            # Evaluate each candidate.
            # (Assuming self.obj returns a tuple; we take the first element.)
            fitnesses = [float(self.obj(sol)) if np.isscalar(self.obj(sol)) else self.obj(sol)[0] for sol in solutions]
            # Update the distribution with the evaluated fitnesses
            self.es.tell(solutions, fitnesses)
            itr += 1
            # Track the best solution so far (for logging and optimality measure)
            current_best = min(fitnesses)
            if current_best < best_f:
                best_f = current_best
                best_x = solutions[np.argmin(fitnesses)]
            # Log outputs (update modOpt outputs)
            self.update_outputs(
                itr=itr,
                x=best_x,  # best_x is a list (the candidate solution)
                obj=best_f,
                opt=best_f,
                time=float(time.time() - start_time)
            )
        self.total_time = time.time() - start_time

        # Store final results
        self.results = {
            'x': best_x,
            'objective': best_f,
            'optimality': best_f,
            'itr': itr,
            'time': self.total_time
        }
        self.run_post_processing()
        return self.results

In [5]:
import random
import numpy as np
from deap import base, creator, tools
from modopt import Optimizer

class CHC(Optimizer):

    def initialize(self):
        """Initialize CHC Algorithm within ModOpt."""
        self.solver_name = 'DEAP_CHC'

        self.obj = self.problem._compute_objective

        # Define hyperparameters
        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default=-4.0, types=float)
        self.options.declare('rangeHigh', default=4.0, types=float)
        self.options.declare('diversity_threshold', default=3.0, types=float)  # Hamming distance threshold
        self.options.declare('restart_fraction', default=0.35, types=float)  # Fraction to reinitialize
        self.options.declare('stagnation_limit', default=10, types=int)  # Restart after N generations of no improvement

        self.options.declare('readable_outputs', types=list, default=[])

        self.available_outputs = {
            'itr': int,
            'obj': float,
            'x': (float, (self.problem.nx,)),
            'opt': float,
            'time': float,
        }

    @staticmethod
    def hux_crossover(ind1, ind2):
        """Half Uniform Crossover (HUX): Swaps 50% of differing genes."""
        diff_indices = [i for i in range(len(ind1)) if ind1[i] != ind2[i]]
        num_swaps = len(diff_indices) // 2  # Swap half of differing bits
        
        for idx in random.sample(diff_indices, num_swaps):
            ind1[idx], ind2[idx] = ind2[idx], ind1[idx]

        return ind1, ind2

    @staticmethod
    def hamming_distance(ind1, ind2):
        """Compute Hamming distance between two individuals."""
        return sum(1 for a, b in zip(ind1, ind2) if a != b)

    def setup(self):
        """Setup CHC Algorithm with DEAP"""
        if not hasattr(creator, "FitnessMin"):
            creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attr_float", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])
        self.toolbox.register("individual", tools.initRepeat, creator.Individual, self.toolbox.attr_float, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", self.hux_crossover)
        self.toolbox.register("select", tools.selBest)

        def modopt_evaluate(individual):
            return (self.problem._compute_objective(individual),)

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        """Solve optimization problem using CHC Algorithm"""
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']
        stagnation_limit = self.options['stagnation_limit']
        diversity_threshold = self.options['diversity_threshold']
        restart_fraction = self.options['restart_fraction']

        obj = self.obj

        start_time = time.time()

        x_k = x * 1.
        f_k = obj(x_k)

        itr = 0
        opt = float('inf')
        stagnation_counter = 0  # Track stagnation

        self.update_outputs(itr=0, x=x_k, obj=f_k, opt=opt, time=time.time() - start_time)

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])

        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        best_ind = tools.selBest(pop, 1)[0]
        best_fitness = best_ind.fitness.values[0]

        while opt > opt_tol and itr < maxiter:
            # Sort by fitness
            pop = tools.selBest(pop, len(pop))
            offspring = pop[:]

            # Apply HUX crossover
            for i in range(0, len(offspring) - 1, 2):
                if self.hamming_distance(offspring[i], offspring[i+1]) >= diversity_threshold:
                    offspring[i], offspring[i+1] = self.toolbox.mate(offspring[i], offspring[i+1])

            # Evaluate new individuals
            for ind in offspring:
                del ind.fitness.values
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # Update population
            pop[:] = tools.selBest(pop + offspring, len(pop))

            # Check best fitness
            new_best_ind = tools.selBest(pop, 1)[0]
            new_best_fitness = new_best_ind.fitness.values[0]

            if new_best_fitness < best_fitness:
                best_fitness = new_best_fitness
                best_ind = new_best_ind
                stagnation_counter = 0  # Reset stagnation counter
            else:
                stagnation_counter += 1

            # **Restart mechanism (Cataclysmic Mutation)**
            if stagnation_counter >= stagnation_limit:
                print("Restarting due to stagnation...")
                num_reset = int(len(pop) * restart_fraction)
                for i in range(num_reset):
                    pop[i] = self.toolbox.individual()
                stagnation_counter = 0

            x_k = np.array(best_ind)
            f_k = best_fitness
            opt = f_k
            itr += 1

            # Log progress
            self.update_outputs(itr=itr, x=x_k.tolist(), obj=f_k, opt=opt, time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        self.run_post_processing()
        return self.results

## Hyperparameter Tuning ##

In [7]:
# Assume Rosenbrock2D_SOO is a modOpt problem that implements _compute_objective, etc.
prob = Rosenbrock2D()
optimizer = CMAES(prob, opt_tol=1e-8, maxiter=1000)
results = optimizer.solve()
optimizer.print_results(summary_table=True)
print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

Setting objective name as "f".
(25_w,50)-aCMA-ES (mu_w=14.0,w_1=14%) in dimension 2 (seed=691247, Mon Mar 10 00:07:56 2025)

	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : CMA-ES
	objective                : 1.6458723201556913e-18
	optimality               : 1.6458723201556913e-18
	itr                      : 41
	time                     : 0.4072725772857666
	total_callbacks          : 4100
	obj_evals                : 4100
	grad_evals               : 0
	hess_evals               : 0
	con_evals                : 0
	jac_evals                : 0
	reused_callbacks         : 0
	out_dir                  : Rosenbrock2D_outputs/2025-03-10_00.07.56.070247
	----------------------------------------------------------------------------------------------------

                          modOpt summary table:                          
         #        itr    

In [37]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args

# Define search space for hyperparameters
space = [
    Real(0.05, 0.5, name='mutationRate'),  # Mutation probability
    Real(0.5, 2.0, name='mutationStd'),  # Crossover probability
    Integer(200, 500, name='initialPopulationSize') # Population size
]

# Objective function: runs GA and returns best fitness
@use_named_args(space)
def objective(mutationRate, mutationStd, initialPopulationSize):

    initialPopulationSize = int(initialPopulationSize)
    
    # Initialize GA with given hyperparameters
    optimizer = SimpleGA(
        problem=Rosenbrock2D(),
        mutationRate=mutationRate,
        mutationStd=mutationStd,
        initialPopulationSize=initialPopulationSize,
        maxiter= 200 
    )

    # Run the optimizer
    results = optimizer.solve()

    # Extract objective value (ensure it's a scalar)
    fitness_value = results['objective']
    if isinstance(fitness_value, (list, np.ndarray)):
        fitness_value = fitness_value[0]

    # Return the negative fitness (since skopt minimizes)
    return -fitness_value

In [ ]:
# Run Bayesian Optimization
res = gp_minimize(objective, space, n_calls=30, initial_point_generator="lhs", random_state=42)

# Best hyperparameters
print("Best Parameters Found:")
print(f"Mutation Rate: {res.x[0]:.3f}")
print(f"Mutation Std: {res.x[1]:.3f}")
print(f"Population Size: {res.x[2]}")
print(f"Best Fitness: {-res.fun:.6f}")  # Convert back to positive fitness

In [8]:
# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

prob = Rosenbrock2D()

optimizer = G3PCX(prob,
                            opt_tol=opt_tol,
                            maxiter=maxiter,
                            readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])


# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

Setting objective name as "f".

	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : DEAP_G3PCX
	objective                : 1.8198770226888457e-06
	optimality               : 1.8198770226888457e-06
	itr                      : 1000
	time                     : 7.210744380950928
	total_callbacks          : 112179
	obj_evals                : 112178
	grad_evals               : 1
	hess_evals               : 0
	con_evals                : 0
	jac_evals                : 0
	reused_callbacks         : 0
	out_dir                  : Rosenbrock2D_outputs/2025-03-10_00.08.13.969689
	----------------------------------------------------------------------------------------------------

                          modOpt summary table:                          
         #        itr              obj              opt             time 
         0          0     4.900000E

In [9]:
# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

prob = Rosenbrock2D()

optimizer = CHC(prob,
                opt_tol=opt_tol,
                maxiter=maxiter,
                readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])


# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

Setting objective name as "f".
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restarting due to stagnation...
Restartin

In [11]:
# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

Pop = 700

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = SimpleGA(prob,
                            opt_tol=opt_tol,
                            maxiter=maxiter,
                            mutationRate = 0.2,
                            mutationStd = 1.485,
                            initialPopulationSize = 354,
                            readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Check first derivatives at the initial guess, if needed
optimizer.check_first_derivatives(prob.x0)

# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

optimizer = DEAPGeneric(prob,
                            opt_tol=opt_tol,
                            maxiter=maxiter,
                            initialPopulationSize = Pop,
                            readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])


# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

Setting objective name as "f".

----------------------------------------------------------------------------
Derivative type | Calc norm  | FD norm    | Abs error norm | Rel error norm 
----------------------------------------------------------------------------

Gradient        | 4.9715e+01 | 4.9715e+01 | 1.0012e-04     | 2.0140e-06    
----------------------------------------------------------------------------


	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : Genetic_Algorithm
	objective                : 0.0017752483181734538
	optimality               : 0.0017752483181734538
	itr                      : 1000
	time                     : 12.378394842147827
	total_callbacks          : 704010
	obj_evals                : 704008
	grad_evals               : 2
	hess_evals               : 0
	con_evals                : 0
	jac_evals                : 0